# Building a Custom Scenario

How to connect a new domain to the Quantum Decision Engine by subclassing
`ScenarioAdapter` and implementing the lifecycle hooks.

**QDE flow:** context → belief state → evolving reasoning → decision

The core engine is domain-agnostic. Scenario adapters bridge the gap between
domain-specific signals and the mathematical framework. Each adapter defines:
1. How context maps to an initial belief state
2. What operators are built from the context signals
3. What constraints apply after evolution
4. How the collapsed action maps to a domain output

This notebook demonstrates:
- **`ScenarioAdapter` subclass** — all four lifecycle hooks implemented
- **`ScenarioRegistry`** — register, list, and retrieve scenarios by name
- **`CoreConfig`** — defining belief state geometry
- **`ScenarioConfig`** — domain metadata
- **End-to-end pipeline** — from raw context to decision using the custom adapter

In [1]:
# Environment check
import atto
print(f"atto version: {atto.__version__}")


atto version: 0.1.0


In [2]:
import numpy as np
from atto.api.scenario import ScenarioAdapter
from atto.api.registry import ScenarioRegistry
from atto.config.core import CoreConfig
from atto.config.scenario import ScenarioConfig
from atto.core.state import AttoState
from atto.core.operator import AttoOperator
from atto.core.dynamics import AttoDynamics
from atto.core.measurement import AttoMeasurement

## 1 — Define the Domain

We will build a **project prioritisation** scenario. A team must decide which
initiative to pursue given competing signals: strategic alignment, resource
availability, expected impact, and urgency.

Four possible outcomes:
- `launch` — commit resources and proceed
- `incubate` — dedicate a small team to explore further
- `defer` — wait for better conditions
- `kill` — abandon the initiative

In [4]:
# Define the belief state geometry
core_config = CoreConfig(
    state_dim=4,
    action_labels=["launch", "incubate", "defer", "kill"],
    use_complex=False,
    normalize_state=True,
)

# Attach domain metadata
scenario_config = ScenarioConfig(
    scenario_name="project_prioritisation",
    metadata={
        "domain": "operations",
        "team": "product",
        "signal_schema": "strategic_alignment,resource_availability,expected_impact,urgency",
    },
)

print(f"State dimension:  {core_config.state_dim}")
print(f"Action labels:    {core_config.action_labels}")
print(f"Scenario name:    {scenario_config.scenario_name}")
print(f"Metadata:         {scenario_config.metadata}")

State dimension:  4
Action labels:    ['launch', 'incubate', 'defer', 'kill']
Scenario name:    project_prioritisation
Metadata:         {'domain': 'operations', 'team': 'product', 'signal_schema': 'strategic_alignment,resource_availability,expected_impact,urgency'}


## 2 — Subclass `ScenarioAdapter`

The adapter defines four lifecycle hooks:

| Hook | Purpose |
|------|--------|
| `map_context(x)` | Translate a context vector into an initial belief state |
| `build_operators(x)` | Build operators from context signals |
| `apply_constraints(state)` | Enforce domain-specific constraints after evolution |
| `decode_decision(action, labels)` | Map action index to domain output |

In [5]:
class ProjectPrioritisationAdapter(ScenarioAdapter):
    """
    Scenario adapter for project prioritisation decisions.

    Context vector layout:
      x[0] = strategic_alignment  (0–1)
      x[1] = resource_availability (0–1)
      x[2] = expected_impact       (0–1)
      x[3] = urgency               (0–1)
    """

    def map_context(self, x):
        """Translate context signals into an initial belief state.

        High strategic alignment and impact → lean toward launch.
        Low resource availability → lean toward defer or kill.
        We construct a soft prior from the signals rather than
        starting from uniform.
        """
        alignment, resources, impact, urgency = x[0], x[1], x[2], x[3]

        # Build unnormalised weights for each outcome
        raw = np.array([
            alignment * impact * resources,          # launch
            alignment * impact * (1 - resources),    # incubate
            (1 - urgency) * (1 - impact),            # defer
            (1 - alignment) * (1 - impact),          # kill
        ])
        raw = np.clip(raw, 0.01, None)  # avoid zero probabilities
        probs = raw / raw.sum()

        return AttoState.from_probabilities(probs)

    def build_operators(self, x):
        """Build operators from context signals.

        Each signal generates a distinct operator:
        1. A phase shift from the alignment/impact balance
        2. An interference gate between launch and defer based on urgency
        3. A signal operator from the full context vector
        """
        alignment, resources, impact, urgency = x[0], x[1], x[2], x[3]

        return [
            # Alignment-impact balance shifts phases
            AttoOperator.phase_shift(4, [
                alignment * 0.5,
                impact * 0.3,
                -(1 - resources) * 0.4,
                -(1 - alignment) * 0.3,
            ]),
            # Urgency creates interference between launch and defer
            AttoOperator.interference(4, i=0, j=2, angle=urgency * 0.8),
            # Full context as a composite signal
            AttoOperator.from_signal(x),
        ]

    def apply_constraints(self, state):
        """Enforce domain constraints after evolution.

        In this scenario: no constraints. The belief state evolves freely.
        In other domains this could enforce regulatory limits, capacity
        constraints, or risk thresholds.
        """
        return state

    def decode_decision(self, action, labels):
        """Map action index to domain output.

        Returns the human-readable label. In richer scenarios this could
        return a structured object with rationale, risk flags, etc.
        """
        return labels[action] if labels else str(action)


print("ProjectPrioritisationAdapter defined")

ProjectPrioritisationAdapter defined


## 3 — Register the Scenario

Registration makes the scenario retrievable by name across notebooks and
applications. This is the standard pattern for reusable scenario management.

In [6]:
registry = ScenarioRegistry()
registry.register("project_prioritisation", ProjectPrioritisationAdapter)

print(f"Registered scenarios: {registry.list_scenarios()}")

# Retrieve by name
adapter = registry.get("project_prioritisation")
print(f"Retrieved: {type(adapter).__name__}")

Registered scenarios: ['project_prioritisation']
Retrieved: ProjectPrioritisationAdapter


## 4 — Run the Pipeline End-to-End

Feed a context vector through the adapter's lifecycle hooks and collapse to a
decision. This mirrors what `AttoEngine` does under the hood.

In [7]:
# Context: high alignment, moderate resources, high impact, high urgency
context = np.array([0.85, 0.50, 0.80, 0.90])

# Step 1: Map context to initial belief state
initial_state = adapter.map_context(context)
print("Step 1 — Initial belief state:")
print(f"  probabilities: {initial_state.probabilities}")
print()

Step 1 — Initial belief state:
  probabilities: [0.46575342 0.46575342 0.02739726 0.04109589]



In [8]:
# Step 2: Build operators from context
operators = adapter.build_operators(context)
print(f"Step 2 — Built {len(operators)} operators from context signals")
print()

Step 2 — Built 3 operators from context signals



In [9]:
# Step 3: Evolve the belief state through the operators
dynamics = AttoDynamics()
for op in operators:
    dynamics.add_operator(op)

evolved = dynamics.evolve(initial_state)

# Also get the full trajectory
trajectory = dynamics.evolve_stepwise(initial_state)
step_names = ["Initial", "Phase shift", "Urgency interference", "Composite signal"]

print("Step 3 — Belief state evolution:")
for name, state in zip(step_names, trajectory):
    entropy = AttoMeasurement.entropy(state)
    print(f"  {name:25s} → P = {state.probabilities}  entropy = {entropy:.4f}")
print()

Step 3 — Belief state evolution:
  Initial                   → P = [0.46575342 0.46575342 0.02739726 0.04109589]  entropy = 0.9415
  Phase shift               → P = [0.46575342 0.46575342 0.02739726 0.04109589]  entropy = 0.9415
  Urgency interference      → P = [0.18433599 0.46575342 0.30881469 0.04109589]  entropy = 1.1616
  Composite signal          → P = [0.06869091 0.16349842 0.21317684 0.55463382]  entropy = 1.1365



In [10]:
# Step 4: Apply constraints
constrained = adapter.apply_constraints(evolved)
print(f"Step 4 — After constraints: {constrained.probabilities}")
print()

Step 4 — After constraints: [0.06869091 0.16349842 0.21317684 0.55463382]



In [11]:
# Step 5: Collapse to a decision
measurement = AttoMeasurement(method="argmax")
decision = measurement.collapse(constrained)

label = adapter.decode_decision(decision.action, core_config.action_labels)

print("Step 5 — Collapsed decision:")
print(f"  action:        {decision.action}")
print(f"  label:         {label}")
print(f"  confidence:    {decision.confidence:.4f}")
print(f"  probabilities: {decision.probabilities}")

Step 5 — Collapsed decision:
  action:        3
  label:         kill
  confidence:    0.5546
  probabilities: [0.06869091 0.16349842 0.21317684 0.55463382]


## 5 — Compare Different Contexts

Run several context vectors through the adapter to see how different signal
combinations produce different decisions.

In [12]:
contexts = {
    "High alignment, high impact, resourced, urgent": [0.9, 0.8, 0.9, 0.9],
    "High alignment, high impact, under-resourced":   [0.9, 0.2, 0.8, 0.5],
    "Low alignment, low impact, not urgent":           [0.2, 0.5, 0.2, 0.1],
    "Mixed signals — moderate everything":             [0.5, 0.5, 0.5, 0.5],
    "High urgency, low alignment":                     [0.2, 0.7, 0.3, 0.9],
}

measurement = AttoMeasurement(method="argmax")

print(f"{'Context':<50} {'Decision':<12} {'Conf':>6}  Probabilities")
print("-" * 100)

for desc, ctx in contexts.items():
    x = np.array(ctx)
    state = adapter.map_context(x)
    ops = adapter.build_operators(x)

    dyn = AttoDynamics()
    for op in ops:
        dyn.add_operator(op)

    evolved = adapter.apply_constraints(dyn.evolve(state))
    d = measurement.collapse(evolved)
    label = adapter.decode_decision(d.action, core_config.action_labels)

    probs_str = "  ".join(f"{p:.3f}" for p in d.probabilities)
    print(f"{desc:<50} {label:<12} {d.confidence:>6.3f}  [{probs_str}]")

Context                                            Decision       Conf  Probabilities
----------------------------------------------------------------------------------------------------
High alignment, high impact, resourced, urgent     kill          0.700  [0.008  0.003  0.289  0.700]
High alignment, high impact, under-resourced       incubate      0.403  [0.247  0.403  0.083  0.268]
Low alignment, low impact, not urgent              kill          0.641  [0.002  0.058  0.300  0.641]
Mixed signals — moderate everything                kill          0.784  [0.013  0.021  0.182  0.784]
High urgency, low alignment                        kill          0.954  [0.005  0.020  0.021  0.954]


## 6 — Order Sensitivity

Demonstrate that changing the operator order for the same context produces
a different result — the non-commutativity that QDE captures naturally.

In [13]:
context = np.array([0.7, 0.4, 0.8, 0.6])
state = adapter.map_context(context)
ops = adapter.build_operators(context)

# Original order
dyn_fwd = AttoDynamics()
for op in ops:
    dyn_fwd.add_operator(op)

# Reversed order
dyn_rev = AttoDynamics()
for op in reversed(ops):
    dyn_rev.add_operator(op)

result_fwd = measurement.collapse(dyn_fwd.evolve(state))
result_rev = measurement.collapse(dyn_rev.evolve(state))

print("Original operator order:")
print(f"  decision:      {adapter.decode_decision(result_fwd.action, core_config.action_labels)}")
print(f"  probabilities: {result_fwd.probabilities}")
print()
print("Reversed operator order:")
print(f"  decision:      {adapter.decode_decision(result_rev.action, core_config.action_labels)}")
print(f"  probabilities: {result_rev.probabilities}")
print()

sensitivity = dyn_fwd.sensitivity(state)
print(f"Order sensitivity: {sensitivity:.4f}")

Original operator order:
  decision:      kill
  probabilities: [0.03862841 0.26113105 0.11965204 0.58058851]

Reversed operator order:
  decision:      incubate
  probabilities: [0.01622117 0.4789725  0.04624383 0.45856249]

Order sensitivity: 0.2612


## Summary

Building a custom scenario requires four steps:

| Step | What to define | Purpose |
|------|---------------|--------|
| 1 | `map_context(x)` | How domain signals become a belief state |
| 2 | `build_operators(x)` | What reasoning steps the signals produce |
| 3 | `apply_constraints(state)` | Domain-specific post-evolution constraints |
| 4 | `decode_decision(action, labels)` | How the collapsed action maps to domain output |

Once defined, register the adapter with `ScenarioRegistry` for reuse:

```python
registry.register("my_scenario", MyAdapter)
adapter = registry.get("my_scenario")
```

The core engine handles all the mathematics — state evolution, interference,
non-commutativity, and measurement. The scenario adapter only needs to define
how domain signals connect to that framework.